In [ ]:
from collections import Counter
import re

# 1. Sample Data
data = [
    "I am sitting in front of Bank",
    "He is sitting in front of Bank",
    "She is sitting in front of Bank",
    "We are sitting in front of Bank",
    "They are sitting in front of Bank",
]

print("--- 1. Extracting Pronouns/Subjects ---")
# Extract the first word of each sentence
for sentence in data:
    words = sentence.split()
    subject = words[0]
    print(f"Sentence: '{sentence}' -> Subject: {subject}")

print("\n--- 2. Finding Common Phrases & Patterns ---")
# Count the occurrences of full sentences to see patterns
sentence_counts = Counter(data)
for sentence, count in sentence_counts.items():
    print(f"'{sentence}': appears {count} time(s)")

print("\n--- 3. Counting Word Frequencies (Vocabulary Analysis) ---")
# Combine all text, lower the case, and count individual words
all_text = " ".join(data).lower()
# Use regex to find all words, ignoring punctuation if any
words_list = re.findall(r"\b\w+b?", all_text)
word_counts = Counter(words_list)

# Display the words from most common to least common
for word, frequency in word_counts.most_common():
    print(f"Word: '{word:<7}' | Count: {frequency}")


In [ ]:
import random

# 1. Sample Data (Training Set)
corpus = [
    "I am sitting in front of Bank",
    "He is sitting in front of Bank",
    "She was sitting in front of Bank",
    "She will be sitting in front of Bank",
    "She will be sitting in front of Bank",
    "We are sitting in front of Bank",
    "They are sitting in front of Bank"
]

# 2. Build a Simple Language Model (Bigram/Context Probabilities)
# This mimics how a decoder learns which word follows a given context.
transitions = {}

for sentence in corpus:
    words = ["<START>"] + sentence.split() + ["<END>"]
    for i in range(len(words) - 1):
        context = words[i]
        next_word = words[i+1]
        if context not in transitions:
            transitions[context] = []
        transitions[context].append(next_word)

# 3. The Autoregressive Decoder Function
def autoregressive_decoder(prompt, max_length=10):
    # Initialize the sequence with the starting prompt
    generated_sequence = prompt.split()
    print(f"Initial Prompt: {generated_sequence}")

    # Autoregressive Loop: Each iteration generates exactly ONE word
    for step in range(max_length):
        # Current context is the last word we generated
        current_word = generated_sequence[-1]

        # Look up what words can follow this context
        choices = transitions.get(current_word, ["<END>"])

        # Predict the next token (Sampling from learned distribution)
        next_token = random.choice(choices)

        # Stop if the model decides the sentence is finished
        if next_token == "<END>":
            print(f"Step {step+1}: Generated '<END>' token. Halting.")
            break

        # CRITICAL AUTOREGRESSIVE STEP:
        # Append the new token back into the input sequence
        generated_sequence.append(next_token)
        print(f"Step {step+1} Output: {' '.join(generated_sequence)}")

    return " ".join(generated_sequence)

# 4. Run the Decoder
print("--- Starting Generation ---")
final_output = autoregressive_decoder("She")
print(f"\nFinal Generated Sentence: \"{final_output}\"")


In [ ]:
!pip install onnxruntime_genai

In [ ]:
import onnxruntime_genai as og
# Load a model (this path works for CPU, GPU, or mobile automatically)
model = og.Model('path/to/model')
tokenizer = og.Tokenizer(model)
# Configure how you want to search
params = og.GeneratorParams(model)
params.set_search_options(max_length=256, batch_size=1)
# The generation loop
generator = og.Generator(model, params)
generator.append_tokens(tokenizer.encode(prompt))
while True:
    generator.generate_next_token()
    if generator.is_done():
        break

    # Decode and print as we go
    token = generator.get_next_tokens()[0]
    print(tokenizer.decode(token), end='', flush=True)

In [ ]:
!pip install gliner

In [3]:
from gliner import GLiNER

# Load a lightweight, highly capable generalist model
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

def protect_pii(text: str, labels: list) -> str:
    """Detects specified PII entities and redacts them from the text."""
    # Predict entities using zero-shot labels
    entities = model.predict_entities(text, labels, threshold=0.4)

    # Sort entities in reverse order to modify text from back to front
    # (prevents index shifting bugs during string manipulation)
    sorted_entities = sorted(entities, key=lambda x: x['start'], reverse=True)

    redacted_text = text
    for entity in sorted_entities:
        start = entity['start']
        end = entity['end']
        label = entity['label'].upper().replace(" ", "_")

        # Replace the PII with a safe placeholder token
        redacted_text = redacted_text[:start] + f"[{label}]" + redacted_text[end:]

    return redacted_text


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

In [5]:
# 1. Chatbot Log Sanitization
text_1 = "My card number is 4111-2222-3333-4444 and the CVV is 123."
labels_1 = ["credit card number", "cvv"]
# Output: "My card number is [CREDIT_CARD_NUMBER] and the CVV is [CVV]."
print(protect_pii(text_1,labels_1))

# 2. Email Ticketing Anonymization
text_2 = "Please send the package to 123 Main St, New York, or call me at 555-0199."
labels_2 = ["home address", "phone number"]

# 3. Call Transcript Redaction
text_3 = "Caller verified identity. Date of birth: August 14, 1985. Mother's maiden name: Smith."
labels_3 = ["date of birth", "mother's maiden name"]

# 4. Contact Form Filtering
text_4 = "Help, I locked myself out. My password is 'Admin2026!' and PIN is 9982."
labels_4 = ["password", "pin"]
# 5. Court Document Anonymization
text_5 = "The minor child, Jane Doe, witnessed the transaction from bank account 0098112-A."
labels_5 = ["name of minor", "witness name", "asset account number"]

# 6. Contract Review Preparation
text_6 = "Signed by Johnathan Ross (Executive VP). Contact via jross@corp-inc.com."
labels_6 = ["executive signature", "personal email address"]

# 7. DSAR (Data Subject Access Request) Processing
text_7 = "Manager Alice Vance discussed performance concerns with HR representative Bob on extension 404."
labels_7 = ["colleague name", "private phone extension"]

# 8. Regulatory Audit Cleansing
text_8 = "Corporate Tax ID EIN 12-3456789 was verified against government forms."
labels_8 = ["employee identification number", "tax identifier"]
# 9. EHR De-identification
text_9 = "Patient admitted on 10/12/2026 under MRN-992384. Fingerprint scan stored in bio-vault."
labels_9 = ["medical record number", "biometric identifier", "date of admission"]

# 10. Clinical Trial Data Sharing
text_10 = "Subject lives in Boston, zip code 02108, and shows positive progression."
labels_10 = ["geographic data", "zip code"]

# 11. Medical Research Paper Scrubbing
text_11 = "Dr. Robert Chen monitored patient J.K. over a three-week clinical period."
labels_11 = ["physician name", "patient initials"]
# 12. Application Log Data Masking
text_12 = "Error 500 from 192.168.1.50 using bearer token eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9."
labels_12 = ["api key", "oauth token", "ip address"]

# 13. Git Repository Scanning
text_13 = "git_config = { 'user': 'dev_admin', 'ssh_key': 'ssh-rsa AAAAB3NzaC1yc2E...' }"
labels_13 = ["username", "password", "private ssh key"]

# 14. Database Migration Anonymization
text_14 = "The user registered a red Ford Mustang, VIN 1FA6P8CF0H5XXXXXX, under her previous name, Miller."
labels_14 = ["maiden name", "vehicle identification number"]

# 15. LLM Prompt Guardrails
text_15 = "Summarize the financial performance of Acme Corp for our client Vanguard Group."
labels_15 = ["corporate identity", "customer account detail"]
# 16. Loan Application Processing
text_16 = "Applicant Social Security Number is 000-12-3456. National Insurance No: QQ123456C."
labels_16 = ["social security number", "national insurance number"]

# 17. Resume/CV Blind Hiring
text_17 = "Sarah Jenkins graduated from MIT in 2018 with a focus on systemic algorithms."
labels_17 = ["candidate name", "gender marker", "graduation year"]

# 18. Expense Report Auditing
text_18 = "Total $450.00 routed from account 44009821. Authorized by manually signed mark: [Image_Ref_01]."
labels_18 = ["personal banking routing number", "personal signature"]

# 19. Background Check Verification
text_19 = "Background clear: No felony offenses found. Credit score evaluated at 740 points."
labels_19 = ["criminal record details", "credit history metrics"]

# 20. Know Your Customer (KYC) Pipelines
text_20 = "Passport No: Z1234567, expiring on 12/2031. Driver's license verified."
labels_20 = ["passport number", "driver's license expiration date"]
print(protect_pii(text_20,labels_20))


[CREDIT_CARD_NUMBER] is [CREDIT_CARD_NUMBER] and the [CVV] is [CVV].
Passport No: [PASSPORT_NUMBER], expiring on [DRIVER'S_LICENSE_EXPIRATION_DATE]. Driver's license verified.


In [6]:
# Choose standard log sanitization setup (Example 12)
log_data = "Unauthorized access attempt by 10.0.4.15 using token xyz123key at 14:00."
target_pii = ["api key", "oauth token", "ip address"]

# Run the protection engine
clean_log = protect_pii(log_data, target_pii)

print("Original:", log_data)
print("Sanitised:", clean_log)
# Sanitised: Unauthorized access attempt by [IP_ADDRESS] using token [API_KEY] at 14:00.


Original: Unauthorized access attempt by 10.0.4.15 using token xyz123key at 14:00.
Sanitised: Unauthorized access attempt by [IP_ADDRESS] using token [OAUTH_TOKEN] at 14:00.
